# Folio — Qwen3-4B T4 training

Select **Runtime → Change runtime type → T4 GPU**, then connect. The notebook checks the actual allocation. Training starts from the pinned base, uses FP16 on T4, masks prompt loss, rejects oversized examples, and saves checkpoints to your Drive. It does not enable Folio AI.

The default data is a synthetic pipeline pilot plus eight individually reviewed public summaries. It is not enough to establish real-user reliability. Public bulk datasets remain outside training. See the included dataset review. You authorize Colab/Drive to store this public/synthetic training package by running its cells; no personal Folio records or credentials are included.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
%pip install -q transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.48.1 'safetensors>=0.4.3'


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose T4 GPU before continuing'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3, 'GiB')
assert 'T4' in torch.cuda.get_device_name(0), 'This notebook was requested for T4. Change the runtime to T4.'
print('T4 training precision: FP16')


In [ ]:
import base64, hashlib, io, zipfile
from pathlib import Path
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='93599443eb7d8d996dff37f5fec4cfeabc41e3f31b25d9e4d2da9f27f6483ca2'
TOOLS=Path('/content/folio-training');TOOLS.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        destination=(TOOLS/item.filename).resolve()
        assert destination.is_relative_to(TOOLS.resolve())
        destination.write_bytes(z.read(item))
print('Extracted verified package:', sorted(p.name for p in TOOLS.iterdir()))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
RUN=Path('/content/drive/MyDrive/FolioTraining')/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN.mkdir(parents=True,exist_ok=False)
print('Checkpoints and reports:', RUN)


In [ ]:
import subprocess
subprocess.run(['python','-m','unittest','discover','-s',str(TOOLS),'-p','test_*.py','-v'],check=True)
RAW_DATA=RUN/'raw-data'
DATA=RUN/'data'
command=['python',str(TOOLS/'prepare_data.py'),'--output',str(RAW_DATA)]
if (TOOLS/'approved-public.jsonl').exists():command+=['--approved',str(TOOLS/'approved-public.jsonl')]
subprocess.run(command,check=True)
subprocess.run(['python',str(TOOLS/'align_runtime_data.py'),'--input',str(RAW_DATA),'--output',str(DATA)],check=True)


## First run: small pipeline test

This trains eight optimizer steps and evaluates the base and adapter. Inspect the outputs and memory use before running the longer experiment. Loss and JSON scores do not prove factual accuracy.

In [ ]:
subprocess.run(['python','-u',str(TOOLS/'train.py'),'--data',str(DATA),'--output',str(RUN/'smoke'),'--steps','8','--accumulation','4','--checkpoint-every','4'],check=True)


In [ ]:
import json
report=json.loads((RUN/'smoke'/'experiment-report.json').read_text())
print(json.dumps({k:v for k,v in report.items() if k not in ('optimized_row_ids','losses')},indent=2))
for case in json.loads((RUN/'smoke'/'adapter-validation.json').read_text())['cases']:
    if not case['structured_fields_match']:
        print(case['kind'], case['output'], case['errors'])


## Longer experiment — run after inspecting the pilot

This is one pass over the current curated pilot mix, starting from the same original base. It remains development training. Add independently authored prompts and reviewed public examples before treating it as product qualification. Run this cell manually when ready.

In [ ]:
rows=json.loads((DATA/'manifest.json').read_text())['splits']['train']['rows']
steps=(rows+3)//4
subprocess.run(['python','-u',str(TOOLS/'train.py'),'--data',str(DATA),'--output',str(RUN/'full'),'--steps',str(steps),'--accumulation','4','--checkpoint-every','25'],check=True)


## Recovering an interrupted run

Keep this notebook and the printed RUN path. Reconnect T4 and repeat dependency/package setup. Do not create new data or delete old outputs. Use the same saved data and a fresh output folder with `--resume-checkpoint` pointing to the last completed checkpoint. Its manifest and optimizer state must match. Download the experiment report and adapter to continue local evaluation; do not enable it in Folio until factual, scope, approval, export, and runtime checks pass.